In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("=" * 80)
print("STEP 1: Job parameters received")
print("=" * 80)
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  table_name        : '{table_name}'")
print(f"  load_type         : '{load_type}'")
print(f"  merge_keys        : '{merge_keys}'")

In [0]:
source_table = f"adb_medilake.bronze.{table_name}"
source_filter = f"landing_timestamp = '{landing_timestamp}'"

print("=" * 80)
print("STEP 2: Reading source data from bronze")
print("=" * 80)
print(f"  Source table : {source_table}")
print(f"  Filter       : {source_filter}")
print(f"  Equivalent query:\n    SELECT * FROM {source_table} WHERE {source_filter}")

bronze_df = spark.read.table(source_table).filter(source_filter)

print(f"  Columns ({len(bronze_df.columns)}): {bronze_df.columns}")

In [0]:
from pyspark.sql import functions as F

target_table = f"adb_medilake.silver.{table_name}"
normalized_load_type = (load_type or "FULL").strip().upper()
merge_key_list = [key.strip() for key in merge_keys.split(",") if key.strip()]

print("=" * 80)
print("STEP 3: Preparing load metadata")
print("=" * 80)
print(f"  Target table      : {target_table}")
print(f"  Raw load_type     : '{load_type}'")
print(f"  Effective type    : {normalized_load_type}" + ("  (defaulted to FULL because load_type was empty)" if not load_type.strip() else ""))
print(f"  Merge keys parsed : {merge_key_list if merge_key_list else 'None'}")

if normalized_load_type == "MERGE":
    if not merge_key_list:
        raise ValueError("merge_keys must be provided when load_type is 'MERGE'.")

    missing_merge_keys = [key for key in merge_key_list if key not in bronze_df.columns]
    if missing_merge_keys:
        raise ValueError(f"merge_keys not found in source data: {missing_merge_keys}")
    print("  Merge key validation passed: all keys exist in source data")

updated_df = (
    bronze_df.withColumn("insert_timestamp", F.current_timestamp())
    .withColumn("update_timestamp", F.current_timestamp())
)

print("  Added audit columns: insert_timestamp, update_timestamp (current_timestamp())")
print(f"  Final source columns ({len(updated_df.columns)}): {updated_df.columns}")

In [0]:
print("=" * 80)
print(f"STEP 4: Writing to silver using load type: {normalized_load_type}")
print("=" * 80)
print(f"  Target table exists: {spark.catalog.tableExists(target_table)}")

if normalized_load_type == "FULL":
    print(f"  Action: OVERWRITE all data in {target_table}")
    print(f"  Equivalent query:\n    INSERT OVERWRITE {target_table} SELECT * FROM <source_dataframe>")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("  FULL load completed")

elif normalized_load_type == "APPEND":
    print(f"  Action: APPEND new data to {target_table}")
    print(f"  Equivalent query:\n    INSERT INTO {target_table} SELECT * FROM <source_dataframe>")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("  APPEND load completed")

elif normalized_load_type == "MERGE":
    if not spark.catalog.tableExists(target_table):
        print(f"  Target table does not exist; creating {target_table} from source data instead of merging")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print("  Initial table creation completed")
    else:
        source_view = "silver_merge_source"
        updated_df.createOrReplaceTempView(source_view)
        print(f"  Registered source data as temp view: {source_view}")

        merge_condition = " AND ".join(
            [f"target.`{key}` = source.`{key}`" for key in merge_key_list]
        )

        update_assignments = ", ".join(
            [
                f"target.`{column}` = source.`{column}`"
                for column in updated_df.columns
                if column != "insert_timestamp"
            ]
        )

        insert_columns = ", ".join([f"`{column}`" for column in updated_df.columns])
        insert_values = ", ".join([f"source.`{column}`" for column in updated_df.columns])

        merge_query = f"""
            MERGE INTO {target_table} AS target
            USING {source_view} AS source
            ON {merge_condition}
            WHEN MATCHED THEN UPDATE SET {update_assignments}
            WHEN NOT MATCHED THEN INSERT ({insert_columns})
            VALUES ({insert_values})
        """

        print(f"  Merge condition    : {merge_condition}")
        print("  Columns updated on match (insert_timestamp preserved):")
        print(f"    {update_assignments}")
        print(f"  Columns inserted when not matched: {insert_columns}")
        print("  MERGE query being executed:")
        print(merge_query)

        merge_result = spark.sql(merge_query)
        print("  MERGE completed. Metrics:")
        for row in merge_result.collect():
            for metric, value in row.asDict().items():
                print(f"    {metric}: {value}")

else:
    print(f"  ERROR: Unsupported load type '{normalized_load_type}'")
    raise ValueError("load_type must be one of: FULL, APPEND, MERGE")

In [0]:
record_count = updated_df.count()

print("=" * 80)
print("STEP 5: Load summary")
print("=" * 80)
print(f"  Source       : adb_medilake.bronze.{table_name} (landing_timestamp = '{landing_timestamp}')")
print(f"  Target       : {target_table}")
print(f"  Load type    : {normalized_load_type}")
print(f"  Records read : {record_count}")
print(f"  Rows in target after load: {spark.table(target_table).count()}")
print("  Exiting notebook with record count")
dbutils.notebook.exit(str(record_count))

In [0]:
print("[STEP 4] Computing processed record count")
record_count = updated_df.count()
print(f"[STEP 4 COMPLETE] Processed record count: {record_count}")
dbutils.notebook.exit(str(record_count))